# 02 — Preprocessing

This notebook turns raw text into the inputs the models need and **persists the
results to `../data/`** so that `03_Modelling.ipynb` can load them without
recomputing anything.

Two artifacts are produced:

1. **Tokenized dataset** (`input_ids`, `attention_mask`) — used to *fine-tune* DistilBERT.
2. **Hidden-state features** (the `[CLS]` embedding per text) — used by the
   *feature-extraction* baselines (XGBoost, Logistic Regression).

Features are extracted for **all three splits** (train, validation **and test**),
so the test set is available for a single, untouched final evaluation later.

## 1. Setup

In [ ]:
# Run once if the packages are missing (e.g. in Colab):
# !pip install -q -r ../requirements.txt

from pathlib import Path

import numpy as np
import tensorflow as tf
from datasets import load_dataset
from transformers import AutoTokenizer, TFAutoModel

In [ ]:
# Configuration shared with the modelling notebook.
MODEL_CKPT = "distilbert-base-uncased"
MAX_LENGTH = 64  # covers virtually all texts (see length analysis in 01_EDA)
DATA_DIR = Path("..") / "data"
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
emotions = load_dataset("emotion", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
emotions

## 2. Tokenization

We tokenize with a **fixed `max_length`** and `padding="max_length"` so every
example has the same, reproducible shape. The original notebook passed
`truncation=True` *without* a `max_length`, which silently disabled truncation.

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )


emotions_encoded = emotions.map(tokenize, batched=True)
emotions_encoded["train"].column_names

Persist the tokenized dataset so the fine-tuning notebook can load it directly.

In [ ]:
encoded_path = DATA_DIR / "emotions_encoded"
emotions_encoded.save_to_disk(str(encoded_path))
print("Saved tokenized dataset to", encoded_path)

## 3. Hidden-state features

For the feature-extraction baselines we run each text through a **frozen**
DistilBERT and keep the last hidden state of the `[CLS]` token as a 768-dim
feature vector.

Unlike the original notebook, `extract_hidden_states` **reuses the already
computed `input_ids`/`attention_mask`** instead of tokenizing a second time.

In [ ]:
bert_model = TFAutoModel.from_pretrained(MODEL_CKPT)


def extract_hidden_states(batch):
    inputs = {
        col: tf.convert_to_tensor(batch[col]) for col in tokenizer.model_input_names
    }
    outputs = bert_model(inputs)
    # [CLS] token (position 0) is the standard sentence representation.
    cls_embedding = outputs.last_hidden_state[:, 0].numpy()
    return {"hidden_states": cls_embedding}


# A fixed batch_size keeps memory bounded and padding consistent.
emotions_hidden = emotions_encoded.map(
    extract_hidden_states, batched=True, batch_size=64
)

In [ ]:
features = {}
for split in ("train", "validation", "test"):
    X = np.array(emotions_hidden[split]["hidden_states"])
    y = np.array(emotions_hidden[split]["label"])
    features[split] = (X, y)
    print(f"{split:>10}: X={X.shape}, y={y.shape}")

## 4. Persist the features

Save the arrays as `.npy` so `03_Modelling.ipynb` can load them instantly.

In [ ]:
split_to_suffix = {"train": "train", "validation": "val", "test": "test"}
for split, suffix in split_to_suffix.items():
    X, y = features[split]
    np.save(DATA_DIR / f"X_{suffix}.npy", X)
    np.save(DATA_DIR / f"y_{suffix}.npy", y)

print("Saved features to", DATA_DIR.resolve())

Done. `03_Modelling.ipynb` now reads `data/emotions_encoded` (for fine-tuning)
and `data/X_*.npy` / `data/y_*.npy` (for the baselines).